# Handling Conflicts in DataFrames

When merging, there may be overlapping column names between the two DataFrames that you are not explicitly joining on. Pandas handles this by appending suffixes (like `_x` and `_y`) to distinguish them. We will see how this works and how to handle merges using multiple columns to avoid ambiguity.

In [17]:
# Import pandas library
import pandas as pd

### Conflicting Column Names

Here we create two DataFrames that both have a `Location` column. When we merge them on `Name`, pandas needs to figure out what to do with the `Location` columns since a person might have different locations in different DataFrames.

In [18]:
# Create staff DataFrame with Location
staff_df = pd.DataFrame([{'Name': 'Kelly', 'Role': 'Director of HR', 
                          'Location': 'State Street'},
                         {'Name': 'Sally', 'Role': 'Course liasion', 
                          'Location': 'Washington Avenue'},
                         {'Name': 'James', 'Role': 'Grader', 
                          'Location': 'Washington Avenue'}])

# Create student DataFrame with Location representing their home/residence
student_df = pd.DataFrame([{'Name': 'James', 'School': 'Business', 
                            'Location': '1024 Billiard Avenue'},
                           {'Name': 'Mike', 'School': 'Law', 
                            'Location': 'Fraternity House #22'},
                           {'Name': 'Sally', 'School': 'Engineering', 
                            'Location': '512 Wilson Crescent'}])

Display the `staff_df`:

In [19]:
# View staff_df
staff_df

,Name,Role,Location
0,Kelly,Director of HR,State Street
1,Sally,Course liasion,Washington Avenue
2,James,Grader,Washington Avenue


Display the `student_df`:

In [20]:
# View student_df
student_df

,Name,School,Location
0,James,Business,1024 Billiard Avenue
1,Mike,Law,Fraternity House #22
2,Sally,Engineering,512 Wilson Crescent


### Suffixes for Conflict Resolution

When we merge on 'Name', the 'Location' column from `staff_df` becomes `Location_x` and from `student_df` becomes `Location_y`. This preserves the data from both tables while keeping column names unique.

In [21]:
# Left join on 'Name'. Notice how 'Location' is duplicated as 'Location_x' (left DF) and 'Location_y' (right DF)
pd.merge(staff_df, student_df, how="left", on="Name")

,Name,Role,Location_x,School,Location_y
0,Kelly,Director of HR,State Street,NaN,NaN
1,Sally,Course liasion,Washington Avenue,Engineering,512 Wilson Crescent
2,James,Grader,Washington Avenue,Business,1024 Billiard Avenue


### Merging on Multiple Columns

What if names are not unique? It's often better to join on multiple columns simultaneously. Here, we split the name into `First Name` and `Last Name`. This way, a join can be highly specific and precise.

In [22]:
# Redefine DataFrames, this time using First Name and Last Name instead of just Name
staff_df = pd.DataFrame([{'First Name': 'Kelly', 'Last Name': 'Desjardins', 
                          'Role': 'Director of HR'},
                         {'First Name': 'Sally', 'Last Name': 'Brooks', 
                          'Role': 'Course liasion'},
                         {'First Name': 'James', 'Last Name': 'Wilde', 
                          'Role': 'Grader'}])
student_df = pd.DataFrame([{'First Name': 'James', 'Last Name': 'Hammond', 
                            'School': 'Business'},
                           {'First Name': 'Mike', 'Last Name': 'Smith', 
                            'School': 'Law'},
                           {'First Name': 'Sally', 'Last Name': 'Brooks', 
                            'School': 'Engineering'}])

By passing a list of column names to the `on` parameter, we force pandas to match records where *both* `First Name` and `Last Name` match. Notice that Sally Brooks is successfully identified as both staff and student, while James Hammond (student) and James Wilde (staff) are treated as separate individuals.

In [24]:
# Outer join on both First Name AND Last Name
pd.merge(staff_df, student_df, how='outer', on=['First Name','Last Name'])

,First Name,Last Name,Role,School
0,James,Hammond,NaN,Business
1,James,Wilde,Grader,NaN
2,Kelly,Desjardins,Director of HR,NaN
3,Mike,Smith,NaN,Law
4,Sally,Brooks,Course liasion,Engineering
